# DESC 3x2pt Static-Probes Figure of Merit - MAF implementation and demo


- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-10
- Context: SCOC (Survey Cadence Optimization Committee) - DESC Task Force metrics, restricted to the DESC metrics only (3x2pt, Weak Lensing, Supernovae)
- This notebook: **3x2pt** (galaxy clustering + weak lensing static-probes Figure of Merit)
- Companion notebooks in this series (`06_MAF_DESC_TaskF`) will cover the **WL** and **SN** DESC Task Force metrics separately.
- Simulation analyzed: `/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db`
- Inspired by `02_MAF/science/DESC/01_sigma8tomography_demo.ipynb` (same repository) and by the official `rubin_sim.maf.batches.science_radar_batch` "Cosmology" group, which is the batch used to populate the DESC 3x2pt entries on the standard MAF show_maf pages. https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/batches/science_radar_batch.py


## Notebook overview

**What is "3x2pt"?** In DESC (LSST Dark Energy Science Collaboration) static-probe cosmology, "3x2pt" refers to the joint analysis of three two-point correlation functions built from a lens (foreground) galaxy sample and a source (background, weak-lensing) galaxy sample, in tomographic redshift bins:

1. **Galaxy clustering** (lens x lens auto-correlation) - constrains galaxy bias and growth of structure.
2. **Cosmic shear** (source x source auto-correlation) - constrains the weak-lensing power spectrum.
3. **Galaxy-galaxy lensing** (lens x source cross-correlation) - breaks degeneracies between bias and amplitude.

Combined, these three probes give one of the tightest static (non-time-domain) cosmological constraints achievable with LSST, usually summarized as a **Figure of Merit (FoM)** on the dark-energy equation-of-state parameters (w0, wa), following the Dark Energy Task Force convention (FoM = 1 / area of the w0-wa confidence ellipse).

**How MAF turns a cadence into a 3x2pt FoM.** Running an actual 3x2pt Fisher-matrix forecast inside MAF for every OpSim run would be far too slow. Instead, `rubin_sim.maf` uses a two-stage, emulator-based approach, exactly analogous to the sigma8-tomography demo of the companion notebook:

1. A per-Healpix-pixel (**parent**) metric, `ExgalM5WithCuts`, computes the coadded, dust-extinction-corrected 5-sigma depth in a chosen band (default `i`) in every sky pixel, after masking pixels that fail quality cuts (high Galactic extinction, insufficient filter coverage, depth below a year-dependent threshold). The result is a Healpix **depth map** restricted to the region of sky usable for static cosmology that year.
2. A **summary** metric reduces that depth map to two survey-level numbers - the effective sky **area** (number of unmasked pixels x pixel area) and the **median depth** over that area - and looks up the corresponding 3x2pt FoM in a pre-computed grid of full DESC Fisher-matrix forecasts (Lochner et al. 2018, arXiv:1808.00006). Two emulator implementations exist in `rubin_sim.maf`:
   - `StaticProbesFoMEmulatorMetricSimple` (`rubin_sim.maf.metrics.cosmology_summary_metrics` https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/metrics/cosmology_summary_metrics.py): a simple bilinear interpolation over a coarse (area, depth) grid, defined only at survey years 1, 3, 6 and 10. This is the version used below (no extra dependencies).
   - `StaticProbesFoMEmulatorMetric` (`rubin_sim.maf.maf_contrib.static_probes_fom_summary_metric` https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/maf_contrib/static_probes_fom_summary_metric.py): a Gaussian-Process emulator (via the `george` package https://github.com/dfm/george) trained on a denser 36-point grid that also varies weak-lensing systematics (multiplicative shear bias, photo-z scatter and bias). This is the version used by the *official* `science_radar_batch` (the batch behind the standard show_maf "Cosmology" pages). See Section 7 below.

This notebook builds the same chain as the official batch (same depth cuts, same `nside=64`, same sky cuts), runs it year by year on the v5.3.6 baseline, and shows:
- the **Healpix sky maps** and **area-weighted histograms** of the usable-depth map for a few sample years (these are produced natively by MAF's own `HealpixSkyMap` / `HealpixHistogram` plotters - i.e. exactly how MAF itself visualizes this metric);
- the resulting **effective area**, **median depth** and **3x2pt FoM** as a function of survey year.


## Simulation and code references
- OpSim run analyzed: `baseline_v5.3.6_10yrs.db` (Rubin baseline v5.3.6, 10-year simulation)
- `rubin_sim.maf` source (main branch, retrieved for this notebook):
  - `rubin_sim/maf/metrics/exgal_m5.py` (`ExgalM5`)
  - `rubin_sim/maf/metrics/weak_lensing_systematics_metric.py` (`ExgalM5WithCuts`, `WeakLensingNvisits`)
  - `rubin_sim/maf/metrics/cosmology_summary_metrics.py` (`StaticProbesFoMEmulatorMetricSimple`)
  - `rubin_sim/maf/maf_contrib/static_probes_fom_summary_metric.py` (`StaticProbesFoMEmulatorMetric`, GP-based)
  - `rubin_sim/maf/batches/science_radar_batch.py` (official "Cosmology" batch definition, function `science_radar_batch`)
- summary.h5 / MAF outputs for standard runs: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/
- Table of simulations: https://usdf-maf.slac.stanford.edu/
- LSST survey strategy : https://github.com/lsst-pst/survey_strategy/

## 1. Imports

In [ ]:
import os
import inspect
from os.path import splitext, basename

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maps
import rubin_sim.maf.metric_bundles as mb

print("rubin_sim version:", rubin_sim.__version__)

### Upstream bug workaround

The installed `rubin_sim.maf.metrics.cosmology_summary_metrics.StaticProbesFoMEmulatorMetricSimple.run()` 
calls `scipy.interpolate.RectBivateateSpline(...)` - this is a **typo in the `rubin_sim` source itself** 
(it should be `RectBivariateSpline`), so the call raises `AttributeError: module 'scipy.interpolate' 
has no attribute 'RectBivateateSpline'` for every year the FoM is actually defined (1, 3, 6, 10). 
This is not something in this notebook; it can be verified directly in 
`rubin_sim/maf/metrics/cosmology_summary_metrics.py` (around line 196).

Without touching the conda environment, we work around it by aliasing the misspelled name onto the 
correct SciPy class in the `scipy.interpolate` module. Because Python modules are singletons (looked 
up once in `sys.modules`), this alias is visible from inside `rubin_sim` as well, even though it does 
`from scipy import interpolate` internally.

In [ ]:
from scipy import interpolate as _scipy_interpolate

if not hasattr(_scipy_interpolate, "RectBivateateSpline"):
    # Typo fix for rubin_sim's cosmology_summary_metrics.StaticProbesFoMEmulatorMetricSimple.run(),
    # which calls the misspelled 'RectBivateateSpline' instead of 'RectBivariateSpline'.
    _scipy_interpolate.RectBivateateSpline = _scipy_interpolate.RectBivariateSpline
    print(
        "Patched scipy.interpolate.RectBivateateSpline -> RectBivariateSpline "
        "(rubin_sim upstream typo workaround)."
    )
else:
    print("scipy.interpolate.RectBivateateSpline already present, no patch needed.")

`I nedded to apply corrections to Correction de cosmology_summary_metrics.py/StaticProbesFoMEmulatorMetricSimple`
in file: `/Users/dagoret/miniconda3/envs/conda_py313_opsim53/lib/python3.13/site-packages/rubin_sim/rubin_sim/maf/metrics/cosmology_summary_metrics.py`, line 196 
and also other bugs



```python
class StaticProbesFoMEmulatorMetricSimple(BaseMetric):
    """
    This calculates the Figure of Merit for the combined
    static probes (3x2pt, i.e., Weak Lensing, LSS, Clustering).
    This FoM is purely statistical and does not factor in systematics.

    Parameters
    ----------
    year : `int`, optional
        The year of the survey to calculate FoM.
        This calibrates expected depth and area.

    Returns
    -------
    result : `float`
        The simple 3x2pt FoM emulator value, for the
        years where the correlation between area/depth and value is defined.

    Notes
    -----
    This FoM is purely statistical and does not factor in systematics.
    The implementation here is simpler than in
    `rubin_sim.maf.mafContrib.StaticProbesFoMEmulatorMetric`, and that
    more sophisticated version should replace this metric.

    This version of the emulator was used to generate the results in
    https://ui.adsabs.harvard.edu/abs/2018arXiv181200515L/abstract

    Note that this is truly a summary metric and should be run on the
    output of Exgalm5_with_cuts.

    """

    def __init__(self, year=10, col=None, **kwargs):

        super().__init__(col=col, **kwargs)
        if col is None:
            self.col = "metricdata"
        self.year = year
        # Set a mask_val so that all metric data is passed
        # for summary calculation, even masked values.
        self.mask_val = -1

    def run(self, data_slice, slice_point=None):
        """
        Args:
            data_slice (ndarray): Values passed to metric by the slicer,
                which the metric will use to calculate metric values
                at each slice_point.
            slice_point (Dict): Dictionary of slice_point metadata passed
                to each metric.

        Returns:
             float: Interpolated static-probe statistical Figure-of-Merit.

        Raises:
             ValueError: If year is not one of the 4 for which a FoM is
             calculated

        """
        nside = hp.npix2nside(len(data_slice))
        # Chop off any outliers
        good_pix = np.where(data_slice[self.col] > 0)[0]
        if np.size(good_pix) == 0:
            return self.badval

        # Calculate area and med depth from
        area = hp.nside2pixarea(nside, degrees=True) * np.size(good_pix)
        median_depth = np.median(data_slice[self.col][good_pix])
        # FoM is calculated at the following values
        if self.year == 1:
            areas = [7500, 13000, 16000]
            depths = [24.9, 25.2, 25.5]
            fom_arr = [
                [1.212257e02, 1.462689e02, 1.744913e02],
                [1.930906e02, 2.365094e02, 2.849131e02],
                [2.316956e02, 2.851547e02, 3.445717e02],
            ]
        elif self.year == 3:
            areas = [10000, 15000, 20000]
            depths = [25.5, 25.8, 26.1]
            fom_arr = [
                [1.710645e02, 2.246047e02, 2.431472e02],
                [2.445209e02, 3.250737e02, 3.516395e02],
                [3.173144e02, 4.249317e02, 4.595133e02],
            ]

        elif self.year == 6:
            areas = [10000, 15000, 20000]
            depths = [25.9, 26.1, 26.3]
            fom_arr = [
                [2.346060e02, 2.414678e02, 2.852043e02],
                [3.402318e02, 3.493120e02, 4.148814e02],
                [4.452766e02, 4.565497e02, 5.436992e02],
            ]

        elif self.year == 10:
            areas = [10000, 15000, 20000]
            depths = [26.3, 26.5, 26.7]
            fom_arr = [
                [2.887266e02, 2.953230e02, 3.361616e02],
                [4.200093e02, 4.292111e02, 4.905306e02],
                [5.504419e02, 5.624697e02, 6.441837e02],
            ]
        else:
            warnings.warn("FoMEmulator is not defined for this year")
            return self.badval

        # Interpolate FoM to the actual values for this sim
        #areas = [[i] * 3 for i in areas]
        #depths = [depths] * 3
        #f = interpolate.RectBivateateSpline(np.ravel(areas), np.ravel(depths), np.ravel(fom_arr))
        #f = interpolate.RectBivariateSpline(np.ravel(areas), np.ravel(depths), np.ravel(fom_arr))
        f = interpolate.RectBivariateSpline(areas,depths,fom_arr,kx=1,ky=1)

        #fom = f(area, median_depth)[0]
        fom = f(area, median_depth)[0,0]
        return fom

```

## 2. Configuration

The path below points directly at the local baseline v5.3.6 OpSim database, as requested. `RUBIN_SIM_DATA_DIR` (used elsewhere in this repository via `get_baseline()`) is left untouched; here we bypass it and analyze this specific file.

In [ ]:
opsim_fname = "/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db"
assert os.path.isfile(opsim_fname), f"OpSim database not found: {opsim_fname}"

run_name = splitext(basename(opsim_fname))[0]
print("run_name:", run_name)

In [ ]:
# Output directories, following the repository convention:
# data_<NN_TAG>/ for MAF outputs (.npz, resultsDb), figs_<NN_TAG>/ for figures (dual PNG + PDF)
NB_TAG = "3X2PTS"
data_dir = f"data_01_{NB_TAG}"
figs_dir = f"figs_01_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

In [ ]:
# Set up the MAF results database (bookkeeping of metric bundles run in this notebook)
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. The MAF metric classes used for the 3x2pt static-probes FoM

We import the classes directly and print their docstrings, so the calculation is fully traceable to the `rubin_sim.maf` source rather than treated as a black box.

In [ ]:
from rubin_sim.maf.metrics.exgal_m5 import ExgalM5
# ExgalM5: coadded 5-sigma depth in one band, corrected for Galactic dust extinction (uses the DustMap).

from rubin_sim.maf.metrics.weak_lensing_systematics_metric import ExgalM5WithCuts
# ExgalM5WithCuts: per-pixel ('parent') metric. Wraps ExgalM5 and additionally MASKS pixels that fail
# the static-science quality cuts: E(B-V) above a threshold, fewer than n_filters bands observed, or
# coadded depth below a year-dependent depth_cut. This is exactly the metric the ExgalM5WithCuts
# docstring says is "required as input for StaticProbesFoMEmulatorMetricSimple" (a 3x2pt FoM emulator).

from rubin_sim.maf.metrics.cosmology_summary_metrics import StaticProbesFoMEmulatorMetricSimple
# StaticProbesFoMEmulatorMetricSimple: 'summary' metric. Reduces the ExgalM5WithCuts Healpix map to two
# numbers (effective area, median depth) and bilinearly interpolates the 3x2pt FoM on a small grid of
# full DESC Fisher-matrix forecasts, separately calibrated for survey years 1, 3, 6 and 10.

print(inspect.getdoc(ExgalM5))
print("-" * 80)
print(inspect.getdoc(ExgalM5WithCuts))
print("-" * 80)
print(inspect.getdoc(StaticProbesFoMEmulatorMetricSimple))

In [ ]:
# %pinfo ExgalM5WithCuts          # uncomment to inspect the full call signature interactively
# %psource ExgalM5WithCuts.run    # uncomment to view the per-pixel cut logic

## 4. Year-dependent configuration (matching the official `science_radar_batch` "Cosmology" group)

The static-science depth cuts below are the same ones used to populate the official MAF "Cosmology" group (`rubin_sim.maf.batches.science_radar_batch`), so that the numbers produced here are directly comparable to the standard show_maf pages for this OpSim run. They follow the LSST Science Requirements Document (SRD) single-visit/coadded-depth expectations per year of survey, in the `i` band, with a small 0.1 mag safety offset.


In [ ]:
bandpass = "i"
nfilters_needed = 6
lim_ebv = 0.2
nside = 64
offset = 0.1

mag_cuts = {
    1: 24.75 - offset,
    2: 25.12 - offset,
    3: 25.35 - offset,
    4: 25.50 - offset,
    5: 25.62 - offset,
    6: 25.72 - offset,
    7: 25.80 - offset,
    8: 25.87 - offset,
    9: 25.94 - offset,
    10: 26.00 - offset,
}
years = sorted(mag_cuts.keys())

# The bilinear-interpolation 3x2pt FoM emulator (StaticProbesFoMEmulatorMetricSimple) is only
# calibrated at these survey years:
fom_defined_years = [1, 3, 6, 10]

pix_area = hp.nside2pixarea(nside, degrees=True)
print(f"nside={nside} -> pixel area = {pix_area:.4f} deg^2")
pd.Series(mag_cuts, name="i-band coadded depth cut (mag)").rename_axis("year")

## 5. Running the metric chain, year by year

For each survey year we:
1. select the visits accumulated up to that year, excluding Deep Drilling Field visits (`scheduler_note not like 'DD%'`);
2. build a `HealpixSlicer` (nside=64) with the Galactic-dust map attached;
3. run `ExgalM5WithCuts` as the per-pixel metric (depth map with quality cuts applied);
4. reduce that map with summary metrics: `MeanMetric`, `MedianMetric`, `RmsMetric`, a `CountRatioMetric` configured to report the **effective area** in deg^2, and `StaticProbesFoMEmulatorMetricSimple` for the **3x2pt FoM** (only meaningful for years 1, 3, 6, 10).

In [ ]:
dustmap = maps.DustMap(nside=nside, interp=False)


def run_3x2pt_year(opsim_fname, run_name, year, out_dir):
    """Run the ExgalM5WithCuts + 3x2pt-FoM-emulator MAF chain for a single survey year.

    Parameters
    ----------
    opsim_fname : str
        Path to the OpSim SQLite database to analyze.
    run_name : str
        Label used for the MAF MetricBundle (not a file path).
    year : int
        Survey year (1-10); sets the night<= cut and the i-band depth cut.
    out_dir : str
        MAF output directory (for the .npz cache and resultsDb bookkeeping).

    Returns
    -------
    bundle : maf.MetricBundle
        The evaluated MetricBundle (Healpix depth map + summary_values dict).
    """
    depth_cut = mag_cuts[year]
    sqlconstraint = "night <= %s and scheduler_note not like 'DD%%'" % (year * 365.25 + 0.5)
    info_label = f"{bandpass} band non-DD year {year}"

    parent_metric = metrics.ExgalM5WithCuts(
        lsst_filter=bandpass,
        n_filters=nfilters_needed,
        extinction_cut=lim_ebv,
        depth_cut=depth_cut,
    )

    summary_metrics = [
        metrics.MeanMetric(),
        metrics.MedianMetric(),
        metrics.RmsMetric(),
        metrics.CountRatioMetric(norm_val=1.0 / pix_area, metric_name="Effective Area (deg)"),
        metrics.StaticProbesFoMEmulatorMetricSimple(year=year, metric_name="3x2ptFoM"),
    ]

    slicer = slicers.HealpixSlicer(nside=nside, use_cache=False)

    bundle = mb.MetricBundle(
        parent_metric,
        slicer,
        sqlconstraint,
        maps_list=[dustmap],
        run_name=run_name,
        info_label=info_label,
        summary_metrics=summary_metrics,
    )
    bd = mb.make_bundles_dict_from_list([bundle])
    bgroup = mb.MetricBundleGroup(bd, opsim_fname, out_dir=out_dir, results_db=resultsDb)
    bgroup.run_all()
    return bundle

In [ ]:
bundles_by_year = {}
for year in years:
    print("Running year", year, "...")
    bundles_by_year[year] = run_3x2pt_year(opsim_fname, run_name, year, data_dir)

## 6. Results table

In [ ]:
rows = []
for year in years:
    sv = bundles_by_year[year].summary_values
    rows.append(
        {
            "year": year,
            "i_depth_cut": mag_cuts[year],
            "effective_area_deg2": sv.get("Effective Area (deg)"),
            "mean_depth": sv.get("Mean"),
            "median_depth": sv.get("Median"),
            "rms_depth": sv.get("Rms"),
            "fom_3x2pt": sv.get("3x2ptFoM") if year in fom_defined_years else np.nan,
        }
    )
results_df = pd.DataFrame(rows).set_index("year")
results_df

In [ ]:
results_csv = os.path.join(data_dir, f"{run_name}_3x2pt_results_by_year.csv")
results_df.to_csv(results_csv)
print("Saved:", results_csv)

## 7. Healpix maps and histograms of the usable extragalactic depth

These are produced with the Healpix slicer's own default plotters, `HealpixSkyMap` and `HealpixHistogram` (`slicer.plot_funcs`) - i.e. this is exactly how MAF itself renders `ExgalM5WithCuts` results. Masked pixels (failing the extinction / filter-coverage / depth cuts) are excluded from both the map and the area-weighted histogram.

In [ ]:
def save_bundle_plots(bundle, year, figs_dir, tag_prefix):
    """Generate and save (PNG + PDF) the native MAF SkyMap and Histogram plots for one bundle."""
    made_plots = bundle.plot(savefig=False)
    saved = []
    for plot_type, fig in made_plots.items():
        if fig is None:
            continue
        base = os.path.join(figs_dir, f"{tag_prefix}_year{year:02d}_{plot_type}")
        fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
        fig.savefig(base + ".pdf", bbox_inches="tight")
        saved.append(base)
        plt.close(fig)
    return saved

In [ ]:
sample_years = [1, 5, 10]
for year in sample_years:
    print(f"--- Year {year}: ExgalM5WithCuts ({bandpass}-band usable depth) ---")
    saved = save_bundle_plots(bundles_by_year[year], year, figs_dir, f"{run_name}_ExgalM5WithCuts")
    for s in saved:
        print("  saved:", s + ".png/.pdf")

In [ ]:
# Display the sky map + histogram for year 10 inline
_ = bundles_by_year[10].plot(savefig=False)
plt.show()

## 8. Evolution of effective area, depth and 3x2pt FoM with survey year

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(7, 10), sharex=True)

axs[0].plot(results_df.index, results_df["effective_area_deg2"], marker="o", color="black")
axs[0].set_ylabel("Effective area\n[deg$^2$]")
axs[0].grid(alpha=0.3)

axs[1].plot(results_df.index, results_df["median_depth"], marker="o", color="blue", label="median")
axs[1].fill_between(
    results_df.index,
    results_df["median_depth"] - results_df["rms_depth"],
    results_df["median_depth"] + results_df["rms_depth"],
    alpha=0.2,
    color="blue",
    label="+/- rms",
)
axs[1].set_ylabel(f"Coadded {bandpass}-band\nExgal depth [mag]")
axs[1].legend()
axs[1].grid(alpha=0.3)

defined = results_df.loc[results_df.index.isin(fom_defined_years)]
axs[2].plot(defined.index, defined["fom_3x2pt"], marker="o", color="red")
axs[2].set_ylabel("3x2pt static-probes\nFoM (simple emulator)")
axs[2].set_xlabel("Survey year")
axs[2].grid(alpha=0.3)

fig.suptitle(f"DESC 3x2pt static-probes metrics - {run_name}")
fig.tight_layout()

base = os.path.join(figs_dir, f"{run_name}_3x2pt_summary_vs_year")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 9. Caveats and the more sophisticated (Gaussian-Process) FoM emulator

- `StaticProbesFoMEmulatorMetricSimple` is a *purely statistical* FoM: it only depends on effective area and median depth, and does not account for weak-lensing systematics (multiplicative shear bias, photometric-redshift scatter and bias), galaxy bias, baryonic feedback, etc. It is only calibrated at years 1, 3, 6 and 10, which is why the FoM curve above has just four points.
- The *official* `rubin_sim.maf.batches.science_radar_batch` "Cosmology" group (the batch behind the standard show_maf "Cosmology" pages for every tracked OpSim run) instead uses `StaticProbesFoMEmulatorMetric` from `rubin_sim.maf.maf_contrib.static_probes_fom_summary_metric`: a Gaussian-Process emulator (via the `george` package) trained on a denser 36-point grid of full DESC Fisher-matrix forecasts that also vary the systematics above (default values provided). It is evaluated at `nside=128` in the official batch and is more expensive to run (fits a GP at each call). It requires `george` and `scikit-learn`, which are not always installed by default (`pip install george`).
- Both emulators are ultimately calibrated on the same underlying DESC static-probes forecasts (Lochner et al. 2018), reduced to functions of (area, depth[, systematics]) only - they are fast surrogates for a full 3x2pt Fisher-matrix analysis, not a re-derivation of the power spectra themselves.
- The DESC 3x2pt *tomographic binning* itself (how galaxies are split into redshift bins for the lens and source samples) is a separate, deeper question, studied e.g. in the LSST-DESC 3x2pt Tomography Optimization Challenge (Zuntz et al. 2021, arXiv:2108.13418); it is not part of what these MAF metrics evaluate.
- For the closely related **weak-lensing-only** proxy metric used by the SCOC WL Task Force (`WeakLensingNvisits`, counting gri visits over the same reduced footprint) and the tomographic clustering/lensing bias metrics (`TomographicClusteringSigma8biasMetric`, `MultibandMeanzBiasMetric`), see the companion `02_MAF/science/DESC/01_sigma8tomography_demo.ipynb` notebook and the forthcoming **WL** notebook in this `06_MAF_DESC_TaskF` series.


## References
- Lochner, M. et al. 2018, "Optimizing LSST Observing Strategy for Dark Energy Science", arXiv:1808.00006 - defines the static-probes (3x2pt) Figure-of-Merit forecast grid used by both emulators. https://arxiv.org/abs/1812.00515
- Zuntz, J. et al. 2021, "The LSST-DESC 3x2pt Tomography Optimization Challenge", arXiv:2108.13418, Open Journal of Astrophysics. https://arxiv.org/abs/2108.13418
- Bianco, F. B. et al. 2022, "Optimization of the Observing Cadence for the Rubin Observatory LSST: A Pioneering Process of Community-Focused Experimental Design", ApJS 258, 1 - SCOC context and the 3x2pt FoM as a collective static-probes metric. : https://arxiv.org/abs/2108.01683
- `rubin_sim.maf` documentation: https://rubin-sim.lsst.io/maf.html
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/metrics/`, `rubin_sim/maf/maf_contrib/`, `rubin_sim/maf/batches/science_radar_batch.py`)
